# Lab 10 — Drift in a RAG system
### *Detect input drift with centroid shift, measure end-to-end accuracy, and recover with a better prompt.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/labs/drift_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
Your retrieval corpus and code stay fixed, but **users change how they ask**. That can shift the **embedding “center of mass”** of incoming questions and, in practice, change how often the model gives **correct** grounded answers—especially when drifted questions carry **false premises** or lots of pasted noise.

You will:
1. Run a **small RAG** pipeline over a policy-style corpus (code provided).
2. Implement **embedding centroid drift** (see `lectures/drift_lecture`) to quantify how far drifted questions moved vs a baseline week.
3. Evaluate **answer accuracy** on a labeled validation set with a simple **LLM-as-judge** (provided).
4. Edit a **system prompt** so the model **does not treat user-stated “facts” as true** unless the retrieved context supports them—then **re-run** evaluation to confirm accuracy improves.

---

## Learning goals
- Compute **centroid shift** between two sets of question strings using mean embeddings.
- Connect **input drift** to **downstream accuracy** (not only retrieval scores).
- Practice mitigation that does **not** require retraining: a **prompt** update, validated on a fixed eval set.



In [1]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.append("/content/main")
from course_utils import (
    drift_length_stats,
    get_text_embedding,
    lab10_setup,
    lab5_generate_answer,
    lab6_build_retriever,
    lab6_get_corpus,
    lab6_rag_retrieve,
)

lab10_setup()
print("✅ Environment ready!")



🔧 Setting up your environment...
  → Installing core packages...
installing mermaid-python
  → Installing additional packages: dspy
installing dspy
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab10_setup complete — ready.
✅ Environment ready!


---TODO

## Pre-Lab Questions
Answer in 1–2 sentences each. (Edit this cell.)

1. What is one reason a deployed RAG system can get worse even if your code and corpus do not change?
2. Name one thing you should *not* log in production.
3. Centroid shift measures semantic drift using mean embeddings. Why might two question sets have **small** centroid shift but **large** accuracy differences (or the reverse)?

Your answers:

1)

2)

3)



---TODO

## Scientific Question & Hypothesis

**Question:** If real user questions **drift** (longer messages, false premises, pasted incident notes), does a **cheap drift metric** (centroid shift) move noticeably, and does **RAG accuracy** on a labeled validation set **drop**? Can a **prompt change** recover accuracy without touching retrieval?

**My hypothesis:**

Write 3–5 sentences here.



## Scientific process plan
- **Observe:** centroid shift on questions; accuracy on the same weeks with the *default* RAG prompt.
- **Intervene:** strengthen the system prompt (your edit) to handle untrusted user premises.
- **Validate:** re-run accuracy on the **drifted** validation set; compare before vs after.



## Part 1 — Corpus & retriever (provided)

We reuse the small **policy / runbook** corpus from Week 6 (`lab6_get_corpus`). Each document is chunked and embedded; `lab6_rag_retrieve` returns the top passages for a query.



In [2]:
# @title Build retriever (provided)
corpus = lab6_get_corpus()

# Drifted queries are longer; keep k moderately high.
retriever = lab6_build_retriever(corpus, chunk_size=60, overlap=15, top_k=4)

demo = lab6_rag_retrieve("Can interns join on-call after onboarding?", retriever)
print("Retrieved passages (preview):")
for i, p in enumerate(demo["passages"][:2], 1):
    print(f"{i}. {p[:200]}...")



Retrieved passages (preview):
1. [policy_oncall::c0] the on call rotation is required for full time engineers and optional for interns interns may join on call only after completing onboarding and receiving manager approval interns s...
2. [policy_access::c0] interns are granted access to internal tools in the first week interns may not access customer production data elevated access requires manager approval...


## Part 2 — Validation sets: same facts, harder questions

**Baseline week** questions are short and neutral. **Drifted week** questions simulate messy real inputs:

- **False premises** (“my manager said interns are automatically on-call…”, “…so prod DB access is fine…”)
- **Buried question** at the end of a long paste

The **gold answers** state what the corpus actually supports. Your RAG system should **contradict** the false premise when the policy disagrees.



In [3]:
# @title Labeled validation sets (provided — do not change gold answers)
BASELINE_VAL = [
    {
        "question": "Can interns join the on-call rotation?",
        "gold_answer": (
            "They may join only after completing onboarding and receiving manager approval, "
            "and they should start with shadow shifts."
        ),
    },
    {
        "question": "Are interns allowed to access customer production data?",
        "gold_answer": "No — interns may not access customer production data.",
    },
    {
        "question": "What should incident responders do during an incident?",
        "gold_answer": (
            "Acknowledge the page, assess severity, mitigate, communicate updates, "
            "and write a postmortem."
        ),
    },
    {
        "question": "What is required for elevated access?",
        "gold_answer": (
            "Elevated access requires manager approval; security training is required for elevated access."
        ),
    },
]

PREAMBLE = (
    "[Pasted incident notes] People were unclear on severity; comms were late. "
    "Assume leadership wants a minimal write-up only. "
)

DRIFTED_VAL = [
    {
        "question": (
            "Our skip-level manager told the team that interns are automatically placed on-call "
            "right after onboarding with no other steps. Is that already allowed?"
        ),
        "gold_answer": (
            "No — interns may join on-call only after onboarding and manager approval, "
            "and they should start with shadow shifts; it is not automatic with no other steps."
        ),
    },
    {
        "question": (
            "Interns get internal tools in the first week, so they can also access customer "
            "production data as long as they are careful, right?"
        ),
        "gold_answer": "No — interns may not access customer production data.",
    },
    {
        "question": (
            "During outages, responders basically only need to mitigate and can skip status updates "
            "and postmortems if things look stable afterward — correct?"
        ),
        "gold_answer": (
            "No — responders should acknowledge the page, assess severity, mitigate, "
            "communicate updates, and write a postmortem."
        ),
    },
    {
        "question": (
            PREAMBLE * 3
            + "Final question only: if nobody complains after mitigation, is the postmortem optional under our policy?"
        ),
        "gold_answer": (
            "No — writing a postmortem is part of the stated incident workflow; "
            "it is not optional just because nobody complained."
        ),
    },
]

len(BASELINE_VAL), len(DRIFTED_VAL)



(4, 4)

## Part 3 — RAG answer + LLM-as-judge (provided)

`rag_answer` stitches retrieval + `lab5_generate_answer`. The **system prompt** is the main knob you will edit.

`answers_match_judge` asks a separate **grader** call: it should return **YES** only if the model answer matches the **gold factual content** (paraphrases OK). The grader is instructed to be **strict** about false premises: if the model agrees with a premise the gold answer rejects, that is **NO**.



In [22]:
# @title RAG + judge (provided)

# Intentionally soft: cooperative tone, weak grounding rules — accuracy often drops on DRIFTED_VAL.
DEFAULT_RAG_SYSTEM_PROMPT = (
    "You are a helpful workplace assistant. "
    "Use the Context when it helps and give a direct, concise answer."
)

# TODO: Part 6: replace this placeholder with a stronger policy-following prompt.
IMPROVED_RAG_SYSTEM_PROMPT = (
    "You are a helpful workplace assistant. "
    "Use the Context when it helps and give a direct, concise answer."
)


def rag_answer(question: str, retriever, system_prompt: str) -> str:
    retrieved = lab6_rag_retrieve(question, retriever)
    context = "\n\n".join(retrieved["passages"])
    return lab5_generate_answer(query=question, context=context, system_prompt=system_prompt)


_JUDGE_SYSTEM = (
    "You compare a model answer to a gold answer. "
    "Reply with exactly one word: YES or NO. "
    "YES only if the model answer matches the factual content of the gold answer "
    "(paraphrases OK). "
    "If the model accepts a false premise that the gold answer rejects, reply NO. "
    "If the model contradicts the gold answer, reply NO."
)


def answers_match_judge(model_answer: str, gold_answer: str) -> float:
    query = (
        "Does the Model Answer match the Gold Answer per my instructions?\n\n"
        f"Gold Answer: {gold_answer}\n\n"
        f"Model Answer: {model_answer}"
    )
    verdict = lab5_generate_answer(
        query=query,
        context="",
        system_prompt=_JUDGE_SYSTEM,
        max_tokens=100,
    )
    v = (verdict or "").strip().upper()
    return 1.0 if v.startswith("YES") else 0.0


def mean_accuracy(rows, retriever, system_prompt: str, sample: int = 1) -> float:
    # sample>1 averages judge noise (costs more API calls).
    scores = []
    for row in rows:
        sub = []
        for _ in range(sample):
            ans = rag_answer(row["question"], retriever, system_prompt=system_prompt)
            sub.append(answers_match_judge(ans, row["gold_answer"]))
        scores.append(float(np.mean(sub)))
    return float(np.mean(scores)) if scores else 0.0



## Part 4 — ✅ You implement: embedding centroid shift

Following the lecture: embed each question, **average** the unit-normalized embedding vectors to get a centroid for each week, then report **cosine distance** between centroids:

$$
\text{shift} = 1 - \cos(\mu_A, \mu_B)
$$

Normalize each text embedding before averaging; then normalize the mean vector before the dot product.

Implement `embedding_centroid_shift`.



In [5]:
# @title TODO: embedding_centroid_shift


def _normalize_vec(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)


def embedding_centroid_shift(questions_a: list[str], questions_b: list[str]) -> float:
    # Cosine distance between mean embedding centroids (1 - cosine similarity).
    raise NotImplementedError("Implement embedding_centroid_shift (see lecture)")



In [ ]:
# @title Compute centroid shift + quick length stats
q_base = [x["question"] for x in BASELINE_VAL]
q_drift = [x["question"] for x in DRIFTED_VAL]

centroid_dist = embedding_centroid_shift(q_base, q_drift)
print("Embedding centroid shift (baseline vs drifted):", round(centroid_dist, 4))
print("Length stats baseline:", drift_length_stats(q_base))
print("Length stats drifted :", drift_length_stats(q_drift))

plt.figure(figsize=(7, 3))
plt.hist([len(q) for q in q_base], bins=8, alpha=0.6, label="baseline")
plt.hist([len(q) for q in q_drift], bins=8, alpha=0.6, label="drifted")
plt.xlabel("question length (chars)")
plt.ylabel("count")
plt.title("Question length distribution (toy monitoring view)")
plt.legend()
plt.tight_layout()
plt.show()



### Reflection (centroid shift)
- Did drifted questions move far in embedding space? Does that match the length histogram?
- Would centroid shift alone be enough to explain *which* answers became wrong?

Write here:



## Part 5 — Measure accuracy before the prompt fix

Run the cell below with `DEFAULT_RAG_SYSTEM_PROMPT`. You should usually see **lower** accuracy on `DRIFTED_VAL` than on `BASELINE_VAL` (stochasticity: set `JUDGE_SAMPLE` > 1 for a smoother estimate).

**Heads-up:** this issues **several** LLM calls (RAG + judge per example).



In [15]:
# @title Accuracy: baseline vs drifted (default prompt)
JUDGE_SAMPLE = 1  # try 2–3 for less noise

acc_base = mean_accuracy(BASELINE_VAL, retriever, DEFAULT_RAG_SYSTEM_PROMPT, sample=JUDGE_SAMPLE)
acc_drift_default = mean_accuracy(DRIFTED_VAL, retriever, DEFAULT_RAG_SYSTEM_PROMPT, sample=JUDGE_SAMPLE)

print(f"Accuracy  BASELINE_VAL (default prompt): {acc_base:.2f}")
print(f"Accuracy  DRIFTED_VAL  (default prompt): {acc_drift_default:.2f}")



Accuracy  BASELINE_VAL (default prompt): 0.75
Accuracy  DRIFTED_VAL  (default prompt): 0.25


## Part 6 — Mitigation: edit `IMPROVED_RAG_SYSTEM_PROMPT`

**Goal:** improve **drifted-week** accuracy **without** changing retrieval, corpus, or hyperparameters.

Ideas:
- Treat user-stated claims as **untrusted** unless supported by Context.
- Answer **only** from Context; reject false premises the Context contradicts.
- If Context is insufficient, refuse or say you cannot confirm.

Edit `IMPROVED_RAG_SYSTEM_PROMPT` in the **RAG + judge** cell, **re-run that cell**, then run the next cell.



In [ ]:
# @title Accuracy on DRIFTED_VAL after your improved prompt
acc_drift_improved = mean_accuracy(DRIFTED_VAL, retriever, IMPROVED_RAG_SYSTEM_PROMPT, sample=JUDGE_SAMPLE)
print(f"Accuracy DRIFTED_VAL (improved prompt): {acc_drift_improved:.2f}")
print(f"Delta vs default on drifted: {acc_drift_improved - acc_drift_default:+.2f}")

### Debug one example (optional)
Inspect retrieval + answers for a single drifted question if your numbers look surprising.



In [20]:
# @title Optional: single-example trace
row = DRIFTED_VAL[3]
for label, prompt in [("default", DEFAULT_RAG_SYSTEM_PROMPT), ("improved", IMPROVED_RAG_SYSTEM_PROMPT)]:
    ans = rag_answer(row["question"], retriever, system_prompt=prompt)
    print("===", label, "===")
    print("Q:", row["question"])
    print("A:", ans)
    print("gold:", row["gold_answer"])
    print("judge:", answers_match_judge(ans, row["gold_answer"]))
    print()



=== default ===
Q: [Pasted incident notes] People were unclear on severity; comms were late. Assume leadership wants a minimal write-up only. [Pasted incident notes] People were unclear on severity; comms were late. Assume leadership wants a minimal write-up only. [Pasted incident notes] People were unclear on severity; comms were late. Assume leadership wants a minimal write-up only. Final question only: if nobody complains after mitigation, is the postmortem optional under our policy? ...
A: Yes, the postmortem is optional if nobody complains after mitigation, as the policy does not mandate it in such cases.
gold: No — writing a postmortem is part of the stated incident workflow; it is not optional just because nobody complained.
judge: 0.0

=== improved ===
Q: [Pasted incident notes] People were unclear on severity; comms were late. Assume leadership wants a minimal write-up only. [Pasted incident notes] People were unclear on severity; comms were late. Assume leadership wants a min

## Part 7 — Ops report (writing)

--TODO


Write 6–10 sentences:
- What drift signal did you use, and what did it show?
- What happened to accuracy (baseline vs drifted vs after prompt)?
- What failure mode did the drifted questions trigger?
- What would you log in production to debug this faster?

Write here:



---TODO

## Results & conclusion
Summarize numbers (centroid shift, accuracies) and whether your hypothesis held.

Write here:



---

## 🧠 AI Usage Log

Document generative-AI assistance if your course requires it.

| Tool Used | Purpose | Prompt / Context | Verification |
|------------|----------|------------------|--------------|
|  |  |  |  |

**Summary:**



In [21]:
# @title ✅ Self-checks (Lab 10)
print("Running checks...")

try:
    s = embedding_centroid_shift(["hello"], ["hello world"])
    assert isinstance(s, (float, np.floating))
    assert 0.0 <= float(s) <= 2.0
    print("✅ embedding_centroid_shift shape OK")
except Exception as e:
    print("❌ embedding_centroid_shift:", e)

try:
    import course_utils

    ref = course_utils.drift_embedding_centroid_shift
    a = ["What is RAG?", "Define drift."]
    b = ["Summarize this long incident paste " * 20, "Users now paste policies at the end."]
    yours = float(embedding_centroid_shift(a, b))
    gold = float(ref(a, b))
    if abs(yours - gold) > 1e-3:
        print("⚠️ embedding_centroid_shift differs from reference by", abs(yours - gold))
    else:
        print("✅ embedding_centroid_shift matches reference helper")
except Exception as e:
    print("⚠️ reference compare skipped:", e)

print("Done.")



Running checks...
❌ embedding_centroid_shift: Implement embedding_centroid_shift (see lecture)
⚠️ reference compare skipped: module 'course_utils' has no attribute 'drift_embedding_centroid_shift'
Done.
